In [10]:
%pip install polars duckdb

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

FOLDER_PROYEK = Path("/home/jovyan/work")
FOLDER_DATA = FOLDER_PROYEK / "data"
FOLDER_RAW = FOLDER_DATA / "raw"

FOLDER_RAW.mkdir(parents=True, exist_ok=True)

print("Folder proyek :", FOLDER_PROYEK)
print("Folder data   :", FOLDER_RAW)

daftar_zip = list(FOLDER_PROYEK.rglob("*EmoTweetID*.zip"))

print("\nFile ZIP yang ditemukan:")

for i, file in enumerate(daftar_zip, start=1):
    print(f"{i}. {file.resolve()}")

Folder proyek : /home/jovyan/work
Folder data   : /home/jovyan/work/data/raw

File ZIP yang ditemukan:


In [2]:
from pathlib import Path

folder = Path("/home/jovyan/work/data/raw")

print("Isi folder data/raw:\n")

for item in folder.rglob("*"):
    if item.is_file():
        print(item.name)
        print("Lokasi:", item.resolve())

Isi folder data/raw:

EmoTweetID-Corpus-TranslatedEN.csv
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Corpus-TranslatedEN.csv
EmoTweetID-Corpus.csv
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Corpus.csv
EmoTweetID-Human-Lexicon-TranslatedEN.csv
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Human-Lexicon-TranslatedEN.csv
EmoTweetID-Human.csv
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Human.csv
EmoTweetID-Lexicon.csv
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Lexicon.csv
TweetID-fastText.zip
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/TweetID-fastText.zip
TweetID-Word2Vec.zip
Lokasi: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/TweetI

In [3]:
from pathlib import Path

DATA_PATH = Path(
    "/home/jovyan/work/data/raw/extracted/"
    "EmoTweetID Indonesian Emotion Tweet Dataset/"
    "EmoTweetID-Corpus.csv"
)

print("Lokasi dataset:", DATA_PATH)
print("Dataset ditemukan:", DATA_PATH.is_file())

if DATA_PATH.is_file():
    print(f"Ukuran: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")
else:
    raise FileNotFoundError("File dataset tidak ditemukan!")

Lokasi dataset: /home/jovyan/work/data/raw/extracted/EmoTweetID Indonesian Emotion Tweet Dataset/EmoTweetID-Corpus.csv
Dataset ditemukan: True
Ukuran: 208.05 MB


In [4]:
import polars as pl

df = pl.scan_csv(
    str(DATA_PATH),
    infer_schema_length=1000
)

print("Nama kolom:")
print(df.collect_schema().names())

print("\nTipe data:")
print(df.collect_schema())

print("\nLima baris pertama:")
df.head(5).collect()

Nama kolom:
['', 'tweet']

Tipe data:
Schema({'': Int64, 'tweet': String})

Lima baris pertama:


,tweet
i64,str
0,"""Eh l nas l watya ely f hayaty …"
1,"""Lihat, kehidupanku jauh lebih …"
2,"""ramai nya dm"""
3,"""Udane kyk aku mengagumi mu, de…"
4,"""FPI Dibubarkan, Kembali Viral …"


In [5]:
jumlah_baris = df.select(pl.len()).collect().item()
jumlah_kolom = len(df.collect_schema().names())

print(f"Jumlah baris  : {jumlah_baris:,}")
print(f"Jumlah kolom  : {jumlah_kolom}")
print(f"Ukuran file   : {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

if jumlah_baris > 1_000_000:
    print("Status: memenuhi syarat lebih dari 1 juta baris.")
elif DATA_PATH.stat().st_size >= 500 * 1024**2:
    print("Status: memenuhi syarat ukuran minimal 500 MB.")
else:
    print("Status: belum memenuhi kriteria ukuran atau jumlah baris.")

Jumlah baris  : 3,126,987
Jumlah kolom  : 2
Ukuran file   : 208.05 MB
Status: memenuhi syarat lebih dari 1 juta baris.


In [6]:
# Menghitung nilai kosong (null) pada setiap kolom
hasil_missing = df.select(
    pl.all().null_count()
).collect()

print("Jumlah missing values per kolom:")
display(hasil_missing)

Jumlah missing values per kolom:


,tweet
u32,u32
0,0


In [7]:
# Mengambil nama kolom teks
kolom_teks = "tweet"

# Pastikan kolom tersebut tersedia
if kolom_teks not in df.collect_schema().names():
    print("Kolom 'tweet' tidak ditemukan.")
    print("Kolom yang tersedia:", df.collect_schema().names())
else:
    hasil_teks_kosong = df.select(
        (
            pl.col(kolom_teks)
            .cast(pl.String)
            .str.strip_chars()
            .eq("")
        )
        .fill_null(False)
        .sum()
        .alias("jumlah_teks_kosong")
    ).collect()

    display(hasil_teks_kosong)

jumlah_teks_kosong
u32
0


In [8]:
kolom_teks = "tweet"

if kolom_teks not in df.collect_schema().names():
    print("Kolom tweet tidak ditemukan.")
else:
    jumlah_duplikat = df.select(
        pl.col(kolom_teks)
        .is_duplicated()
        .sum()
        .alias("jumlah_duplikat")
    ).collect().item()

    print(f"Jumlah tweet duplikat: {jumlah_duplikat:,}")

Jumlah tweet duplikat: 0


In [9]:
kolom_teks = "tweet"

statistik_teks = df.select(
    pl.col(kolom_teks).str.len_chars().min().alias("minimum_karakter"),
    pl.col(kolom_teks).str.len_chars().mean().alias("rata_rata_karakter"),
    pl.col(kolom_teks).str.len_chars().median().alias("median_karakter"),
    pl.col(kolom_teks).str.len_chars().max().alias("maksimum_karakter")
).collect()

display(statistik_teks)

minimum_karakter,rata_rata_karakter,median_karakter,maksimum_karakter
u32,f64,f64,u32
1,60.375637,43.0,1078


In [10]:
import duckdb

con = duckdb.connect()

path_sql = DATA_PATH.resolve().as_posix().replace("'", "''")

query = f"""
SUMMARIZE
SELECT *
FROM read_csv_auto('{path_sql}')
"""

hasil_summarize = con.execute(query).df()

display(hasil_summarize)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,column0,BIGINT,0,3921972,3453436,2226287.1794174393,996863.5583616694,1406303,2234156,3081956,3126988,0.0
1,column1,VARCHAR,! ! 107 soal dah slese malem ini akhirnya bisa...,~~~~~~~~~~~,3152545,NaN,NaN,NaN,NaN,NaN,3126988,0.0


In [12]:
ukuran_mb = DATA_PATH.stat().st_size / (1024 ** 2)

ringkasan_profiling = pl.DataFrame({
    "Metrik": [
        "Nama dataset",
        "Nama file",
        "Ukuran file (MB)",
        "Jumlah baris",
        "Jumlah kolom",
        "Nama kolom",
        "Memenuhi syarat lebih dari 1 juta baris"
    ],
    "Hasil": [
        "EmoTweetID",
        DATA_PATH.name,
        f"{ukuran_mb:.2f}",
        f"{jumlah_baris:,}",
        str(jumlah_kolom),
        ", ".join(df.collect_schema().names()),
        "Ya" if jumlah_baris > 1_000_000 else "Tidak"
    ]
})

display(ringkasan_profiling)

Metrik,Hasil
str,str
"""Nama dataset""","""EmoTweetID"""
"""Nama file""","""EmoTweetID-Corpus.csv"""
"""Ukuran file (MB)""","""208.05"""
"""Jumlah baris""","""3,126,987"""
"""Jumlah kolom""","""2"""
"""Nama kolom""",""", tweet"""
"""Memenuhi syarat lebih dari 1 j…","""Ya"""


In [13]:
from pathlib import Path

FOLDER_OUTPUT = Path("/home/jovyan/work/output")
FOLDER_OUTPUT.mkdir(parents=True, exist_ok=True)

ringkasan_profiling.write_csv(
    FOLDER_OUTPUT / "ringkasan_profiling.csv"
)

hasil_summarize.to_csv(
    FOLDER_OUTPUT / "duckdb_summarize.csv",
    index=False
)

print("Hasil profiling berhasil disimpan.")
print("Lokasi output:", FOLDER_OUTPUT.resolve())

Hasil profiling berhasil disimpan.
Lokasi output: /home/jovyan/work/output


In [14]:
from pathlib import Path

# Mengambil hasil pemeriksaan data
nama_kolom = df.collect_schema().names()
ukuran_mb = DATA_PATH.stat().st_size / (1024 ** 2)

laporan = f"""
# Laporan Data Profiling EmoTweetID

## 1. Informasi Dataset
- Nama file: {DATA_PATH.name}
- Ukuran file: {ukuran_mb:.2f} MB
- Jumlah baris: {jumlah_baris:,}
- Jumlah kolom: {jumlah_kolom}
- Nama kolom: {', '.join(nama_kolom)}

## 2. Pemeriksaan Awal
- Pemeriksaan struktur dataset telah dilakukan menggunakan Polars.
- Pemeriksaan missing values telah dilakukan.
- Pemeriksaan duplikasi telah dilakukan.
- Statistik awal panjang teks telah dihitung.
- Profiling kolom menggunakan DuckDB SUMMARIZE telah dilakukan.

## 3. Kesimpulan Awal
Dataset telah diperiksa dari sisi struktur dan karakteristik awal.
Hasil pemeriksaan kualitas data akan digunakan sebagai dasar
untuk proses cleaning pada Notebook 2.
"""

print(laporan)

folder_output = Path("/home/jovyan/work/output")
folder_output.mkdir(parents=True, exist_ok=True)

(folder_output / "laporan_profiling.md").write_text(
    laporan,
    encoding="utf-8"
)

print("\nLaporan disimpan di:", folder_output / "laporan_profiling.md")


# Laporan Data Profiling EmoTweetID

## 1. Informasi Dataset
- Nama file: EmoTweetID-Corpus.csv
- Ukuran file: 208.05 MB
- Jumlah baris: 3,126,987
- Jumlah kolom: 2
- Nama kolom: , tweet

## 2. Pemeriksaan Awal
- Pemeriksaan struktur dataset telah dilakukan menggunakan Polars.
- Pemeriksaan missing values telah dilakukan.
- Pemeriksaan duplikasi telah dilakukan.
- Statistik awal panjang teks telah dihitung.
- Profiling kolom menggunakan DuckDB SUMMARIZE telah dilakukan.

## 3. Kesimpulan Awal
Dataset telah diperiksa dari sisi struktur dan karakteristik awal.
Hasil pemeriksaan kualitas data akan digunakan sebagai dasar
untuk proses cleaning pada Notebook 2.


Laporan disimpan di: /home/jovyan/work/output/laporan_profiling.md


In [15]:
from pathlib import Path

folder_output = Path("/home/jovyan/work/output")

print("Daftar hasil profiling:")

for file in sorted(folder_output.glob("*")):
    if file.is_file():
        print(f"- {file.name} ({file.stat().st_size:,} byte)")

Daftar hasil profiling:
- duckdb_summarize.csv (351 byte)
- laporan_profiling.md (666 byte)
- ringkasan_profiling.csv (197 byte)
